# Kinematic analysis (speed, distance, time series)

Computes kinematic metrics from the per-larva position CSVs produced by the
tracker, and generates plots/tables (average speed, speed over time, total
distance).

Input:  `tracking/<session>/<experiment>/larva_positions_<larva>.csv`
Output: `kinematic_metrics/<session>/<experiment>/`


## 1. Imports

In [ ]:
import os
import csv
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Base paths
BASE_TRACKING_PATH = Path("tracking")
BASE_OUTPUT_PATH = Path("kinematic_metrics")

## 2. Data-selection configuration

In [ ]:
# ============================================================================
# MAPPING DICTIONARIES
# ============================================================================

SESSIONS = {
    1: "01",
    2: "02"
}

# Experiments for session 01
EXPERIMENTS_01 = {
    1: "control",
    2: "naklucie",
    3: "pbs",
    4: "coli_2x10_8",
    5: "coli_5x_10_7",
    6: "coli_5x_10_8"
}

# Experiments for session 02
EXPERIMENTS_02 = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x10_7",
    5: "coli_5x10_8"
}

In [ ]:
# ============================================================================
# DATA-SELECTION FUNCTIONS
# ============================================================================

def display_sessions():
    """Print the available sessions."""
    print("\n=== Available sessions ===")
    for num, name in SESSIONS.items():
        print(f"{num}. Session {name}")
    print("==========================\n")

def display_experiments(session):
    """Print the available experiments for a session."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    print(f"\n=== Experiments for session {session} ===")
    for num, name in experiments.items():
        print(f"{num}. {name}")
    print("="*35 + "\n")

def get_session():
    """Read the session number from the user."""
    display_sessions()
    while True:
        try:
            session_num = int(input('Enter session number (1-2): '))
            if session_num in SESSIONS:
                return SESSIONS[session_num]
            else:
                print('Error: choose 1 or 2')
        except ValueError:
            print('Error: enter an integer')

def get_experiment(session):
    """Read the experiment from the user."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    display_experiments(session)
    while True:
        try:
            exp_num = int(input(f'Enter experiment number (1-{len(experiments)}): '))
            if exp_num in experiments:
                return experiments[exp_num]
            else:
                print(f'Error: choose a number between 1 and {len(experiments)}')
        except ValueError:
            print('Error: enter an integer')

def get_larva_number():
    """Read the larva number from the user."""
    while True:
        try:
            larva_num = int(input('Enter larva number (1-6) or 0 for all: '))
            if 0 <= larva_num <= 6:
                return larva_num
            else:
                print('Error: choose a number between 0 and 6')
        except ValueError:
            print('Error: enter an integer')

def get_output_dir():
    """Return the output folder for the current session/experiment."""
    output_dir = BASE_OUTPUT_PATH / SESSION / EXPERIMENT
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

In [ ]:
# ============================================================================
# DATA SELECTION - RUN THIS CELL
# ============================================================================

SESSION = get_session()
EXPERIMENT = get_experiment(SESSION)
LARVA_NUM = get_larva_number()  # 0 = all

print("\n" + "="*50)
print("SELECTED:")
print(f"  Session: {SESSION}")
print(f"  Experiment: {EXPERIMENT}")
print(f"  Larva: {'all (1-6)' if LARVA_NUM == 0 else LARVA_NUM}")
print(f"  Output folder: kinematic_metrics/{SESSION}/{EXPERIMENT}/")
print("="*50)

## 3. Data loading

In [ ]:
# ============================================================================
# DATA-LOADING FUNCTIONS
# ============================================================================

def get_larva_file(session, experiment, larva_num, prefer_angles=False):
    """Return the path to a larva CSV (prefers the basic file without _with_angles)."""
    exp_path = BASE_TRACKING_PATH / session / experiment

    if prefer_angles:
        angles_file = exp_path / f"larva_positions_{larva_num}_with_angles.csv"
        if angles_file.exists():
            return angles_file

    basic_file = exp_path / f"larva_positions_{larva_num}.csv"
    if basic_file.exists():
        return basic_file

    return None

def load_larva_data(filepath):
    """Load larva position data from a CSV file."""
    x_positions, y_positions = [], []
    with open(filepath, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            x_positions.append(float(row['X']))
            y_positions.append(float(row['Y']))
    return x_positions, y_positions

def load_selected_data():
    """Load data for the selected parameters."""
    data = []
    larvae_to_load = [LARVA_NUM] if LARVA_NUM != 0 else range(1, 7)

    for larva_num in larvae_to_load:
        filepath = get_larva_file(SESSION, EXPERIMENT, larva_num)
        if filepath and filepath.exists():
            x_pos, y_pos = load_larva_data(filepath)
            data.append((EXPERIMENT, larva_num, x_pos, y_pos))
            print(f"  [ok] Loaded: larva {larva_num} ({filepath.name})")
        else:
            print(f"  [x] No file for larva {larva_num}")

    return data

print(f"\nLoading data for {SESSION}/{EXPERIMENT}...")
DATA = load_selected_data()
print(f"\nLoaded {len(DATA)} larva(e).")

## 4. Computation functions

In [ ]:
# ============================================================================
# COMPUTATION FUNCTIONS
# ============================================================================

# Configuration constants
SCALE = 0.2  # mm/px - pixel-to-millimeter conversion factor
MIN_MOVEMENT_THRESHOLD = 0.0  # mm - no micro-movement filtering (MIN=0 matches YOLO+ByteTrack and reference data)
MAX_MOVEMENT_THRESHOLD = 5.0  # mm - max movement threshold (filters detection jumps/artifacts)
# 5 mm = 25 px at scale 0.2 - corresponds to 125 mm/s, extremely fast for a larva
ARTIFACT_PROXIMITY = 5  # frames - movements near artifacts are ignored

# Consistent color palette for all plots
# Larva 1: blue, 2: orange, 3: green, 4: purple, 5: red, 6: yellow
LARVA_COLORS = ['#0000FF', '#FFA500', '#008000', '#800080', '#FF0000', '#FFD700']

def _calculate_steps(x_positions, y_positions):
    """
    Compute all steps and their distances.
    Returns a list of distances in mm and a set of artifact indices.
    """
    steps = []
    artifact_indices = set()

    for i in range(len(x_positions) - 1):
        x1, y1 = x_positions[i], y_positions[i]
        x2, y2 = x_positions[i + 1], y_positions[i + 1]
        step_dist_mm = np.sqrt((x2 - x1)**2 + (y2 - y1)**2) * SCALE
        steps.append(step_dist_mm)

        if step_dist_mm > MAX_MOVEMENT_THRESHOLD:
            artifact_indices.add(i)

    return steps, artifact_indices

def _is_near_artifact(idx, artifact_indices, proximity=ARTIFACT_PROXIMITY):
    """Check whether an index is near an artifact."""
    for offset in range(-proximity, proximity + 1):
        if (idx + offset) in artifact_indices:
            return True
    return False

def calculate_speed(x_positions, y_positions, fps=25):
    """
    Compute instantaneous speed from positions.

    Speed = step_distance / step_time, where step_time = 1/fps (0.04 s at 25 fps).

    Filtering:
    - movements outside [MIN, MAX] thresholds = noise/artifacts
    - movements near artifacts (+/-5 frames) = related to multi-detection

    Returns:
        List of instantaneous speeds [mm/s] per step.
    """
    steps, artifact_indices = _calculate_steps(x_positions, y_positions)
    time_step = 1.0 / fps
    speeds = []

    for i, step_dist_mm in enumerate(steps):
        is_valid = (MIN_MOVEMENT_THRESHOLD <= step_dist_mm <= MAX_MOVEMENT_THRESHOLD)
        is_clean = not _is_near_artifact(i, artifact_indices)

        if is_valid and is_clean:
            speed = step_dist_mm / time_step
        else:
            speed = 0.0

        speeds.append(speed)

    return speeds

def calculate_distance(x_positions, y_positions):
    """
    Compute the total traveled distance (same filtering as calculate_speed).

    Returns:
        Total distance in mm.
    """
    steps, artifact_indices = _calculate_steps(x_positions, y_positions)
    distance = 0.0

    for i, step_dist_mm in enumerate(steps):
        is_valid = (MIN_MOVEMENT_THRESHOLD <= step_dist_mm <= MAX_MOVEMENT_THRESHOLD)
        is_clean = not _is_near_artifact(i, artifact_indices)

        if is_valid and is_clean:
            distance += step_dist_mm

    return distance

## 5. Kinematic plots

### 5.1 Average speeds

In [ ]:
# ============================================================================
# PLOT + TABLE: AVERAGE SPEEDS
# ============================================================================

def plot_average_speeds(data, save=True):
    if not data:
        print("No data to plot")
        return

    output_dir = get_output_dir()

    names = []
    avg_speeds = []
    larva_nums = []

    for exp, larva_num, x_pos, y_pos in data:
        speeds = calculate_speed(x_pos, y_pos)
        avg_speed = np.mean(speeds)
        names.append(f"Larva {larva_num}")
        avg_speeds.append(avg_speed)
        larva_nums.append(larva_num)

    sorted_indices = np.argsort(avg_speeds)[::-1]
    sorted_names = [names[i] for i in sorted_indices]
    sorted_speeds = [avg_speeds[i] for i in sorted_indices]
    sorted_larva_nums = [larva_nums[i] for i in sorted_indices]
    sorted_colors = [LARVA_COLORS[(ln-1) % len(LARVA_COLORS)] for ln in sorted_larva_nums]

    # ==================== PLOT ====================
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(sorted_names, sorted_speeds, color=sorted_colors, zorder=2)

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.2f} mm/s',
                ha='center', va='bottom', fontsize=10)

    ax.set_title('Average movement speed per larva', fontsize=14)
    ax.set_xlabel('Larva', fontsize=12)
    ax.set_ylabel('Speed (mm/s)', fontsize=12)
    ax.grid(axis='y', zorder=0)
    plt.tight_layout()

    if save:
        plt.savefig(output_dir / 'avg_speed_plot.png', dpi=150)

    plt.show()

    # ==================== TABLE ====================
    overall_avg = np.mean(avg_speeds)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')

    table_data = [['Larva', 'Average speed [mm/s]']]
    for name, speed in zip(sorted_names, sorted_speeds):
        table_data.append([name, f'{speed:.2f}'])
    table_data.append(['All larvae', f'{overall_avg:.2f}'])

    cell_colors = [['lightgrey', 'lightgrey']]
    cell_colors += [['lavender', 'lavender'] for _ in sorted_names]
    cell_colors += [['lightblue', 'lightblue']]

    table = ax.table(cellText=table_data, loc='center', cellColours=cell_colors)
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 1.5)

    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_text_props(fontweight='bold')
        if i == len(table_data) - 1:
            cell.set_text_props(fontweight='bold', color='black')

    if save:
        plt.savefig(output_dir / 'avg_speed_table.png', dpi=150, bbox_inches='tight')

    plt.show()

    print(f"\nSummary:")
    print(f"   Mean: {overall_avg:.2f} mm/s")
    print(f"   Min: {np.min(avg_speeds):.2f} mm/s")
    print(f"   Max: {np.max(avg_speeds):.2f} mm/s")

# plot_average_speeds(DATA)

### 5.2 Speed over time (time series)

In [ ]:
# ============================================================================
# PLOT + TABLE: SPEED TIME SERIES
# ============================================================================

def plot_speed_timeseries(data, segment_duration_sec=300, fps=25, save=True):
    if not data:
        print("No data to plot")
        return

    output_dir = get_output_dir()
    batch_size = segment_duration_sec * fps

    # ==================== PLOT ====================
    plt.figure(figsize=(12, 6))

    all_timeseries = []

    for idx, (exp, larva_num, x_pos, y_pos) in enumerate(data):
        speeds = calculate_speed(x_pos, y_pos, fps)
        avg_speeds = [np.mean(speeds[i:i + batch_size])
                      for i in range(0, len(speeds), batch_size)]
        time_min = np.arange(len(avg_speeds)) * (segment_duration_sec / 60)

        all_timeseries.append((larva_num, avg_speeds, time_min))

        plt.plot(time_min, avg_speeds, '-o',
                 color=LARVA_COLORS[(larva_num-1) % len(LARVA_COLORS)],
                 label=f"Larva {larva_num}",
                 markersize=4)

    plt.title('Average larva speed across the whole recording', fontsize=14)
    plt.xlabel('Time [min]', fontsize=12)
    plt.ylabel('Speed [mm/s]', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    plt.tight_layout()

    if save:
        plt.savefig(output_dir / 'speed_timeseries_plot.png', dpi=150)

    plt.show()

    # ==================== TABLE ====================
    max_segments = max(len(ts[1]) for ts in all_timeseries)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('tight')
    ax.axis('off')

    header = ['Time [min]'] + [f'L {ts[0]} [mm/s]' for ts in all_timeseries]
    table_data = [header]

    for seg_idx in range(max_segments):
        time_val = seg_idx * (segment_duration_sec / 60)
        row = [f'{time_val:.0f}']
        for larva_num, avg_speeds, time_min in all_timeseries:
            if seg_idx < len(avg_speeds):
                row.append(f'{avg_speeds[seg_idx]:.2f}')
            else:
                row.append('-')
        table_data.append(row)

    n_cols = len(header)
    cell_colors = [['lightgrey'] * n_cols]
    cell_colors += [['lavender'] * n_cols for _ in range(max_segments)]

    table = ax.table(cellText=table_data, loc='center', cellColours=cell_colors)
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.3)

    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_text_props(fontweight='bold')

    if save:
        plt.savefig(output_dir / 'speed_timeseries_table.png', dpi=150, bbox_inches='tight')

    plt.show()

# plot_speed_timeseries(DATA)

### 5.3 Traveled distances

In [ ]:
# ============================================================================
# PLOT + TABLE: TRAVELED DISTANCES
# ============================================================================

def plot_distances(data, save=True):
    if not data:
        print("No data to plot")
        return

    output_dir = get_output_dir()

    names = []
    distances = []
    larva_nums = []

    for exp, larva_num, x_pos, y_pos in data:
        dist = calculate_distance(x_pos, y_pos)
        names.append(f"Larva {larva_num}")
        distances.append(dist)
        larva_nums.append(larva_num)

    sorted_indices = np.argsort(distances)[::-1]
    sorted_names = [names[i] for i in sorted_indices]
    sorted_distances = [distances[i] for i in sorted_indices]
    sorted_larva_nums = [larva_nums[i] for i in sorted_indices]
    sorted_colors = [LARVA_COLORS[(ln-1) % len(LARVA_COLORS)] for ln in sorted_larva_nums]

    # ==================== PLOT ====================
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(sorted_names, sorted_distances, color=sorted_colors, zorder=2)

    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.0f} mm',
                ha='center', va='bottom', fontsize=10)

    ax.set_title('Total distance traveled by each larva', fontsize=14)
    ax.set_xlabel('Larva', fontsize=12)
    ax.set_ylabel('Distance [mm]', fontsize=12)
    ax.grid(axis='y', zorder=0)
    plt.tight_layout()

    if save:
        plt.savefig(output_dir / 'distances_plot.png', dpi=150)

    plt.show()

    # ==================== TABLE ====================
    total_distance = np.sum(distances)
    avg_distance = np.mean(distances)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')

    table_data = [['Larva', 'Distance [mm]']]
    for name, dist in zip(sorted_names, sorted_distances):
        table_data.append([name, f'{dist:.2f}'])
    table_data.append(['Total', f'{total_distance:.2f}'])
    table_data.append(['Mean', f'{avg_distance:.2f}'])

    cell_colors = [['lightgrey', 'lightgrey']]
    cell_colors += [['lavender', 'lavender'] for _ in sorted_names]
    cell_colors += [['lightblue', 'lightblue'], ['lightblue', 'lightblue']]

    table = ax.table(cellText=table_data, loc='center', cellColours=cell_colors)
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 1.5)

    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_text_props(fontweight='bold')
        if i >= len(table_data) - 2:
            cell.set_text_props(fontweight='bold', color='black')

    if save:
        plt.savefig(output_dir / 'distances_table.png', dpi=150, bbox_inches='tight')

    plt.show()

    print(f"\nSummary:")
    print(f"   Total: {total_distance:.0f} mm")
    print(f"   Mean: {avg_distance:.0f} mm")

# plot_distances(DATA)

### 5.4 Movement trajectories

In [ ]:
# ============================================================================
# PLOT: MOVEMENT TRAJECTORIES (optional)
# ============================================================================

# def plot_trajectories(data, save=True):
#     if not data:
#         print("No data to plot")
#         return
#
#     output_dir = get_output_dir()
#     n_larvae = len(data)
#     n_cols = min(3, n_larvae)
#     n_rows = (n_larvae + n_cols - 1) // n_cols
#
#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
#     axes = axes.flatten() if n_larvae > 1 else [axes]
#     colors = plt.cm.viridis(np.linspace(0, 1, 100))
#
#     for idx, (exp, larva_num, x_pos, y_pos) in enumerate(data):
#         ax = axes[idx]
#         n_points = len(x_pos)
#         for i in range(n_points - 1):
#             color_idx = int((i / n_points) * 99)
#             ax.plot(x_pos[i:i+2], y_pos[i:i+2], color=colors[color_idx], linewidth=0.5, alpha=0.7)
#         ax.scatter(x_pos[0], y_pos[0], c='green', s=100, marker='o', label='Start', zorder=5)
#         ax.scatter(x_pos[-1], y_pos[-1], c='red', s=100, marker='x', label='End', zorder=5)
#         ax.set_title(f'Larva {larva_num}', fontsize=12)
#         ax.set_xlabel('X [mm]')
#         ax.set_ylabel('Y [mm]')
#         ax.set_aspect('equal')
#         ax.legend(loc='upper right', fontsize=8)
#         ax.grid(True, alpha=0.3)
#
#     for idx in range(n_larvae, len(axes)):
#         axes[idx].set_visible(False)
#
#     plt.suptitle('Larva movement trajectories', fontsize=14, y=1.02)
#     plt.tight_layout()
#     if save:
#         plt.savefig(output_dir / 'trajectories_plot.png', dpi=150, bbox_inches='tight')
#     plt.show()

# plot_trajectories(DATA)

## 6. Generate all plots

In [ ]:
# ============================================================================
# GENERATE ALL PLOTS AT ONCE
# ============================================================================

def generate_all_plots(data, save=True):
    output_dir = get_output_dir()

    print("=" * 50)
    print(f"GENERATING PLOTS: {SESSION}/{EXPERIMENT}")
    print(f"Output folder: {output_dir}")
    print("=" * 50)

    print("\n1/3 Average speeds (plot + table)...")
    plot_average_speeds(data, save=save)

    print("\n2/3 Speed time series (plot + table)...")
    plot_speed_timeseries(data, save=save)

    print("\n3/3 Traveled distances (plot + table)...")
    plot_distances(data, save=save)

    print("\n" + "=" * 50)
    print("ALL PLOTS GENERATED!")
    if save:
        print(f"  Saved in: {output_dir}")
    print("=" * 50)

generate_all_plots(DATA)